# Step 6 &mdash; Evaluation & Per-Class Thresholding

**Goal:** turn the model's per-class sigmoid scores into binary multi-label predictions while controlling the precision&ndash;recall trade-off for each class separately.

## Why per-class thresholding

A fixed 0.5 threshold assumes each class's score distribution is centered around 0.5. With heavy class imbalance, rare classes typically peak well below 0.5 and would be missed entirely under a uniform threshold. Per-class thresholds, tuned on the **validation set only**, let each class operate at its own optimal F1 point.

## 1. Collect probabilities on a held-out loader

In [ ]:
import numpy as np
import torch

@torch.no_grad()
def collect_probs(model, loader, device):
    model.eval()
    all_y, all_p = [], []
    for tile, global_img, y in loader:
        tile = tile.to(device, non_blocking=True)
        global_img = global_img.to(device, non_blocking=True)
        logits = model(tile, global_img)
        all_y.append(y.numpy())
        all_p.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(all_y), np.concatenate(all_p)

## 2. Per-class threshold search (F1-optimal)

Grid-search each class independently over `[0.05, 0.95]` in 0.05 steps and pick the threshold that maximizes the per-class binary F1. Classes with no positive samples in the search set fall back to a fixed default (0.5).

In [ ]:
from sklearn.metrics import f1_score

def apply_thresholds(y_prob, thresholds):
    if np.isscalar(thresholds):
        return (y_prob >= float(thresholds)).astype(np.int32)
    thr = np.asarray(thresholds, dtype=np.float32).reshape(1, -1)
    return (y_prob >= thr).astype(np.int32)


def find_per_class_thresholds(y_true, y_prob, fixed_th=0.5,
                              th_min=0.05, th_max=0.95, th_step=0.05):
    grid = np.arange(th_min, th_max + 1e-9, th_step, dtype=np.float32)
    C = y_true.shape[1]
    best = np.full(C, float(fixed_th), dtype=np.float32)

    for c in range(C):
        if y_true[:, c].sum() == 0:
            continue  # no positives -> keep default
        yt = y_true[:, c]
        yp = y_prob[:, c]
        best_f1, best_th = -1.0, float(fixed_th)
        for th in grid:
            f1 = f1_score(yt, apply_thresholds(yp, th), average="binary", zero_division=0)
            if f1 > best_f1:
                best_f1, best_th = f1, float(th)
        best[c] = best_th
    return best

**Important:** thresholds are tuned on the **validation set**, then frozen and applied as-is to the **test set**. Tuning on test would leak label information and inflate the reported metrics.

## 3. Metric computation

We report three aggregate metrics:

| Metric | Threshold-dependent? | What it measures |
|--------|----------------------|------------------|
| micro-F1 | yes | Instance-level prediction accuracy |
| macro-F1 | yes | Class-balanced F1 (equal weight per class) |
| mAP (macro) | **no** | Ranking quality, independent of threshold |

`mAP` is our "threshold-free" anchor: it reports the area under each class's PR curve, averaged over classes. Comparing macro-F1 against mAP tells us how much performance is on the table from thresholding alone.

In [ ]:
from sklearn.metrics import (
    precision_recall_fscore_support,
    average_precision_score,
)
import pandas as pd

def compute_metrics(y_true, y_prob, thresholds, label_cols):
    y_pred = apply_thresholds(y_prob, thresholds)
    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    p, r, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    ap = average_precision_score(y_true, y_prob, average=None)
    mAP = float(np.nanmean(ap))

    per_class = pd.DataFrame({
        "class": label_cols,
        "precision": p, "recall": r, "f1": f1,
        "support_pos": support, "ap": ap,
    })
    summary = {"micro_f1": float(micro), "macro_f1": float(macro), "mAP_macro": mAP}
    return summary, per_class

## 4. Final results on the test set

| Model (Dual-Stream) | Thresholding | micro-F1 | macro-F1 | mAP (macro) |
|---------------------|--------------|----------|----------|-------------|
| ResNet50V2          | Fixed 0.5    | 0.7372   | 0.6691   | 0.7061 |
| EfficientNetV2-S    | Fixed 0.5    | **0.7713** | 0.6888 | 0.7112 |
| **SwinV2-Tiny**     | Per-class    | 0.7698   | **0.7505** | **0.7805** |

### Key takeaways

- **SwinV2-Tiny wins on macro-F1 and mAP.** With class imbalance, those are the metrics that matter; SwinV2's window+shifted attention generalizes better to under-represented defect types.
- **EfficientNetV2-S is competitive on micro-F1** but its mAP is meaningfully lower &mdash; ranking quality on rare classes is weaker.
- **Thresholding mainly affects F1**, not mAP: across both CNN baselines, switching from fixed-0.5 to per-class optimization moves micro-F1 by &asymp;3 points but leaves mAP essentially unchanged.

### Hardest classes (SwinV2-Tiny, test set)

Class names are anonymized to comply with the NDA on the original label taxonomy.

| Class | Positives in test | F1 | AP |
|-------|------------------|----|----|
| wear pattern A (fine surface mark) | 56 | 0.400 | 0.345 |
| wear pattern B (sub-mm rupture) | 32 | 0.545 | 0.294 |
| wear pattern C (layered deposit) | 48 | 0.571 | 0.712 |
| wear pattern D (coating loss) | 16 | 0.667 | 0.802 |
| wear pattern E (large rupture) | 96 | 0.696 | 0.742 |

These are the classes future iterations should target with extra labeled data and/or class-focused augmentation.